In [ ]:
## import numpy as np
import pickle as pkl
import os
from dotenv import load_dotenv
load_dotenv()
from matplotlib import pyplot as plt
cot = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/FewShotCOTCLUTRR_Mon_Feb__3_16.37.46_2025_iter0'

cot = np.load(open(cot, 'rb'), allow_pickle=True)

plt.rcParams['font.size'] = 12

In [ ]:
temp_outs_str

In [ ]:
USER_PATH = '/home/XXXX/XXXX/fs_backup_feb13/'
import json
rels = ['aunt', 'brother', 'brother-in-law', 'daughter', 'daughter-in-law','father', 'father-in-law', 'granddaughter', 'grandfather', 'grandmother', 'grandson', 'mother', 'mother-in-law', 'nephew','niece', 'sister', 'sister-in-law', 'son', 'son-in-law', 'uncle']

from datasets import load_from_disk ; ds = load_from_disk(USER_PATH + 'LLM-project/clutrr_clean/dataset_fixed_gpt4o_graph_search/gen_train234_test2to10/test/')
dd = ds.to_pandas().to_dict('records')
ds = []
import random
for d in dd:
    ds.append({})
    ds[-1]['id'] =d['id']
    ds[-1]['context'] = d['clean_story']
    ds[-1]['query'] = d['query']
    if len(d['graph_search_result']) == 0:
        del ds[-1]
        continue
    if random.random() < 0.5:
        ds[-1]['label'] = d['graph_search_result'][0]
        ds[-1]['gt'] = 'true'
    else:
        random.shuffle(rels)
        for rel in rels:
            if rel not in d['graph_search_result'][0]:
                ds[-1]['label'] = rel
                ds[-1]['gt'] = 'false'
                break
        # ds[-1]['label'] = 
    # print(ds[-1])

json.dump(ds,open(USER_PATH + 'SAT-LM/data/new_clutrr_test.json', 'w'))
    

In [ ]:
ds[0]

In [ ]:
len(cot)

In [ ]:
import shutil
import os 

def get_bb(file, del_sols=None):
    bb = {'pos':  [], 'neg': []}
    
    files = ['/'.join(file.split('/')[:-1]) + '/pos_' + file.split('/')[-1], '/'.join(file.split('/')[:-1]) + '/neg_' + file.split('/')[-1] ]
    for i in range(len(files)):
        file = files[i]
        shutil.copy(file, '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
        if not del_sols==None:
            if 'pos' in file:
                if 'neg' in file:
                    print('l. 416 uh oh')
                      
                ds = del_sols['pos']
            elif 'neg' in file:
                ds = del_sols['neg']
            for sol in ds:
                add_clause('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
                cf = open(f'/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]), 'a')
                write_str = '\n'
                for lit in sol:
                    write_str += str(-lit) + ' '
                # write_str += '0'
                cf.write(write_str)
                cf.close()
        # print('running cadical')
        os.system("timeout 5000 /home/XXXX/XXXX/fs_backup_feb13/LLM-project/cadiback/cadiback " + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]) + '> '  + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone")
        #   
        bbone= open('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone", 'r')
        lines = bbone.readlines()
        #   
        for line in lines:
            if line.startswith('b'):
                #   
                lits = line.split(' ')[1:]
                for lit in lits:
                    lit = lit.strip()
                    if lit == '0':
                        continue
                    lit = int(lit)
                    if 'pos' in file:                                
                        if 'neg' in file:
                            print('l. 447 uh oh')
                              
                        bb['pos'].append(lit)
                    elif 'neg' in file:
                            bb['neg'].append(lit)

    return bb


# nameed=False
# c = open(c, 'r')
# cr = csv.reader(c)
# noisy_data = ['clutrr33.cnf']
noisy_data = []
mistr_data = []
c = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_clutrr_csvs/solver_finished.csv'
import csv
import json
dataset = '/home/XXXX/XXXX/fs_backup_feb13/SAT-LM/data/clutrr_test.json'
with open(dataset, 'r') as df:
    data = json.loads(df.read())

task = 'clutrr'
missed=False
c = open(c, 'r')
cr = csv.reader(c)
names = []
all_outs = {}
missed_list = []
labels = {}
for row in cr:
    if row[2] == 'SAT' and row[3] == 'SAT':
        cnf = open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_clutrr/neg_'+row[1]).readlines()[0].strip('\n')
        num_clause = int(cnf.split(' ')[-1])
        if row[1] in noisy_data or row[1] in mistr_data:
            continue
        # if num_clause > 500:
            # continue
        names.append(row[1])
        labels[row[1]] = data[int(row[1].split('clutrr')[1].split('.')[0])]['label']
#   
preds = {}
labels = {}
c = open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/clutrr_labels.csv', 'r')
cr = csv.reader(c)
for row in cr:
    if not os.path.exists('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_clutrr/neg_'+row[0][:-2]+'cnf'):
        continue
    cnf = open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_clutrr/neg_'+row[0][:-2]+'cnf').readlines()[0].strip('\n')
    num_clause = int(cnf.split(' ')[-1])
    if row[1] in noisy_data or row[1] in mistr_data:
        continue
    # if num_clause > 500:
        # continue
    if row[0][:-2]+'cnf' not in names:
        continue
    labels[row[0][:-2]+'cnf'] = row[1].lower()
    



In [ ]:
print(len(names), len(labels))

In [ ]:
cot[i]

In [ ]:
for key, value in labels.items():
    labels[key] = value.strip(' ')

In [ ]:
value

In [ ]:
int(row[1].split('clutrr')[1].split('.cnf')[0])

In [ ]:
i = 0
cot_acc = 0
cot_preds = {}
for name in names:
    if cot[i] == labels[name].strip(' '):
        cot_acc += 1
        cot_preds[key] = True
    else:
        cot_preds[key] = False
    i += 1
print(cot_acc)

In [ ]:
326/(len(names)-500)

In [ ]:
few_shot = "Facts:n[Nancy] likes to cut the hair of her daughter [Heidi].\n[Heidi]'s sister [Lorraine] went to beauty school and taught them all how to cut hair expertly. " + \
            "\nHere are some additional facts and rules we\'ve found:\nNancy is the mother of Lorraine\n If Heidi is the sister of Lorraine and Heidi is the daughter of Nancy then Nancy is the mother of Lorraine.\n" + \
            "Question: Is the following statement true: \n\"[Lorraine] is [Nancy]\'s daughter\"\nAnswer: Let\'s think step by step. \n1. We have already found that Nancy is the mother of Lorraine.\n2. If Nancy is the mother of Lorraine, then Lorraine is the daughter of Nancy.\nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts:\n[Dale] and his sister [Nancy] are decorating for a party.\n[Nancy]'s daughter [Louise] thinks the party will be fun.\n" + \
            "Here are some additional facts and rules we\'ve found:\nDale is the uncle of Louise. If Nancy is the sister of Dale and Nancy is the mother of Louise then Dale is the uncle of Louise.\n" + \
            "Question: Is the following statement true: \n\"[Louise] is not [Dales]\'s niece\"\n" + \
            "Answer: Let\'s think step by step. 1. We are given that Dale is the uncle of Louise.\n2.If Dale is the uncle of Louise, then Louise is the niece of Dale.\nTherefore, the answer is No, the statement is not true.\n***\n" + \
            "Facts: \n[Lillian] and her sister [Nancy] are the only children in their family. \n[Lillian]'s biggest accomplishment is raising her son [Douglas]. " + \
            "\nHere are some additional facts and rules we\'ve found:\nLillian is the sister of Nancy. \nIf Nancy is the sister if Lillian then Lillian is the sister of Nancy.\n" + \
            "Question: Is the following statement true: \n\"[Douglas] is [Nancy]\'s nephew\"\nAnswer: Let\'s think step by step. \n1. [Douglas] is [Lillian]\'s son. \n2. [Nancy] is [Lillian]\'s sister. " + \
            "3\n. [Douglas] is [Nancy]\'s nephew. \nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts: \n[Ashley] liked to go to the park with her granddaughter [Charlotte]. \n[Dale], [Charlotte]'s father, like to take her to the movies instead. " + \
            "\nHere are some additional facts and rules we\'ve found:\nDale is the son of Ashley. If Dale is father of Charlotte and Ashley is the grandmother of Charlotte then Dale is the son of Ashley.\n" + \
            "Question: Is the following statement true: \n\"[Ashley] is not [Dale]\'s mother\"\nAnswer: Let\'s think step by step. \n1. We are given that Dale is the son of Ashley. \n2. If Dale is the son of Ashley, then Ashley is the mother of Dale. " + \
            "\nTherefore, the answer to the question is No, the statement is ot true.\n***\n"

ans = few_shot + 'a;sldkfj;alskdjf***'
print(ans.split('***')[4])

In [ ]:
a = 'grandson_of_james\'_sibling_James_Donald_'
split = a.split('_')
rel_str = ''
for a in split[:-1]:
    rel_str += a + '-'
rel_str = rel_str[:-1]
print(rel_str)

In [ ]:
import copy
import json
# dataset = '/home/XXXX/XXXX/fs_backup_feb13/SAT-LM/data/clutrr_test.json'
dataset = '/home/XXXX/XXXX/fs_backup_feb13/SAT-LM/data/clutrr_test.json'

with open(dataset, 'r') as df:
    data = json.loads(df.read())

# labels = {}

# for i in range(len(data)):
#     labels['clutrr' + str(i) + '.cnf'] = data[i]['gt']
cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/FewShotCOTCLUTRR_Fri_Jan_31_19.25.43_2025_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/LOT_pronto_preds_70BTue_Apr_22_10.55.31_2025_iter'
# cot_iter=  '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/mistral_FewShotCOTCLUTRR_Tue_Apr_29_12.52.15_2025_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/mistral_FewShotCOTPRONTO_Wed_Apr_30_11.26.49_2025_iter'
cot_pred = []
cot_pred_list = []
cot_accs = []
pfx = '/home/XXXX/'
files = ['XXXX/fs_backup_feb13/LLM-project/preds/LOT_clutrr_8B_preds_iter6_[0]', 'XXXX/fs_backup_feb13/LLM-project/preds/LOT_clutrr_8B_preds_iter8_[4]', 'XXXX/fs_backup_feb13/LLM-project/preds/LOT_clutrr_8B_preds_iter0_[1]', 'XXXX/fs_backup_feb13/LLM-project/preds/LOT_clutrr_8B_preds_iter5_[0]', 'XXXX/fs_backup_feb13/LLM-project/preds/LOT_clutrr_8B_preds_iter1_[1]'] 
for i in range(20):
    cot = np.load(open(cot_iter + str(i), 'rb'))
    # cot = np.load(open(pfx + files[i], 'rb'))
    # cot = np.load(open(cot_iter + str(i) + '_[0, 1, 2]', 'rb'))

    cot_acc = 0
    cot_preds = {}
    cot_preds_list = []
    j = 0
    for name in names:
        # print(value, [i])
        
        # name = names[j]
        # if j not in cot.keys(): continue
        if cot[j] == labels[name].strip(' '):
            cot_acc += 1
            cot_preds[name] = True
            cot_preds_list.append(1)
        else:
            cot_preds[name] = False
            cot_preds_list.append(0)
        j += 1
    print(cot_acc)
    cot_accs.append(cot_acc)
    cot_pred.append(copy.deepcopy(cot_preds))
    cot_pred_list.append(copy.deepcopy(cot_preds_list))

In [ ]:
cot

In [ ]:
labels[name]

In [ ]:
n_votes = []
sc_pred = {}
for i in range(len(cot_pred_list[0])):
    n_votes.append(0)
    for j in range(len(cot_pred_list)):
    # for j in range(5):
        n_votes[-1] += cot_pred_list[j][i]
sc_acc = 0
for key, value in cot_pred[0].items():
    tmp = 0
    for j in cot_pred:
        tmp+= j[key]
    if tmp >=len(cot_pred_list)/2+1: 
        sc_pred[key] = 1
        sc_acc += 1
    
    else: sc_pred[key]=0

In [ ]:
len(cot_pred_list)

In [ ]:
n_votes[0]

In [ ]:
np.ceil(1.5)

In [ ]:
len(cot_pred_list)

In [ ]:
np.unique(n_votes, return_counts=True)

In [ ]:
print(np.sum(np.where(np.array(n_votes) >= np.ceil(len(cot_pred_list)/2+0.5), 1, 0) ))
# print(np.sum(np.where(np.array(n_votes) >= 3, 1, 0) ))
# print('sc acc:',np.sum(np.where(np.array(n_votes) >= 3, 1, 0) )/len(cot))

print('sc acc:',np.sum(np.where(np.array(n_votes) >= (np.ceil(len(cot_pred_list)/2+0.5)), 1, 0) )/len(cot))
print('cot acc:', np.mean(cot_accs)/len(cot))

In [ ]:
n_votes


In [ ]:
from sklearn.utils import resample
from scipy.stats import wilcoxon

In [ ]:
print(outs_str)

In [ ]:


temp_outs_str = '/home/XXXX/XXXX/'
outs = pkl.load(open(temp_outs_str, 'rb'))
outs  = temp_outs
bs_outs_acc = []
outs_pred = {}
outs_acc = 0
num_trues = 0
for key, value in outs.items():
    if len(value[1]['neg']) == 0 and labels[key].strip(' ') == 'false':
        outs_pred[key] = True
        outs_acc += 1
    elif len(value[1]['pos']) == 0 and labels[key].strip(' ') == 'true':
        outs_pred[key] = True
        outs_acc += 1
    else:
        outs_pred[key] = False
    if labels[key] == 'true':
        num_trues += 1
outs_acc /= len(outs_pred.keys())
outs_pred_val = np.array(list(outs_pred.values()))

for i in range(len(outs_pred)):
    bs_outs_acc.append(np.sum(resample(outs_pred_val, n_samples=86))/86)
bs_sc = []
bs_sc_acc = []
for i in range(len(outs)):
    bs_sc.append(resample(n_votes[:len(outs)], n_samples=86))
    bs_sc_acc.append(np.sum(np.where(np.array(bs_sc[-1]) >=(np.ceil(len(cot_pred_list)/2+0.5)), 1, 0) )/len(bs_sc[-1]))
# outs['clutrr545.cnf'][1]
print(outs_acc)
print(outs_acc*len(outs_pred.keys()))
print(len(outs))

In [ ]:
print(np.mean(bs_sc_acc))

In [ ]:
outs['clutrr123.cnf'][1]
labels[key]
key = 'clutrr123.cnf'
value = outs[key]
if len(value[1]['neg']) == 0 and labels[key].strip(' ') == 'false':
        print('yippie')
print(value[1]['pos'])

In [ ]:
print(wilcoxon(np.array(bs_outs_acc) - np.array(bs_sc_acc), alternative='greater'))

In [ ]:
d = np.random.binomial( 900, 0.5, 86)/900

In [ ]:
from scipy import stats

confidence_level=0.95
d = d
ci = stats.t.interval(confidence_level, df=len(d)-1, loc=np.mean(d), scale=np.std(d, ddof=1) / np.sqrt(len(d)))
print(ci)

In [ ]:
np.mean(bs_sc_acc)

In [ ]:
 

plt.hist(n_votes, bins=[0,1,2,3,4,5])

In [ ]:
few_shot = "Facts:\n[Nancy] likes to cut the hair of her daughter [Heidi].\n[Heidi]'s sister [Lorraine] went to beauty school and taught them all how to cut hair expertly. " + \
            "\n\nHere are some additional facts we\'ve found:\n[Nancy] is the mother of [Lorraine]\n" + \
            "Question: Is the following statement true: \n\"[Lorraine] is [Nancy]\'s daughter\"\n" + \
            "Answer:\nLet\'s think step by step.  \n1. [Heidi] is the sister of [Lorraine]\n2. [Heidi] is the daughter of [Nancy]\n3. If [Heidi] is the sister of [Lorraine] and [Heidi] is the daughter of [Nancy] then [Nancy] is the mother of [Lorraine].\n4. If [Nancy] is the mother of [Lorraine], then [Lorraine] is the daughter of [Nancy].\nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts:\n[Dale] and his sister [Nancy] are decorating for a party.\n[Nancy]'s daughter [Louise] thinks the party will be fun.\n" + \
            "\nHere are some additional facts we\'ve found:\n[Dale] is the uncle of [Louise]\n" + \
            "Question: Is the following statement true: \n\"[Louise] is not [Dales]\'s niece\"\n" + \
            "Answer: Le\'s think step by step. \n1. [Nancy] is the sister of [Dale]. \n2. [Nancy] is the mother of [Louise]\n3. If [Nancy] is the sister of [Dale] and [Nancy] is the mother of [Louise] then [Dale] is the uncle of [Louise].\n4.If [Dale] is the uncle of [Louise], then [Louise] is the niece of [Dale].\nTherefore, the answer is No, the statement is not true.\n***\n" + \
            "Facts: \n[Lillian] and her sister [Nancy] are the only children in their family. \n[Lillian]'s biggest accomplishment is raising her son [Douglas]. " + \
            "\n\nHere are some additional facts we\'ve found:\n[Lillian] is the sister of [Nancy]\n" + \
            "Question: Is the following statement true: \n\"[Douglas] is [Nancy]\'s nephew\"\n" + \
            "Answer:\nLet\'s think step by step. \n1. [Douglas] is [Lillian]\'s son. \n2. [Nancy] is [Lillian]\'s sister. " + \
            "\n3. If [Douglas] is the son of [Lillian] and [Lillian] is the sister of [Nancy] then [Douglas] is the nephew of [Lillian]. \nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts: \n[Ashley] liked to go to the park with her granddaughter [Charlotte]. \n[Dale], [Charlotte]'s father, like to take her to the movies instead. " + \
            "\n\nHere are some additional facts we\'ve found:\n[Dale] is the son of [Ashley].\n" + \
            "Question: Is the following statement true: \n\"[Ashley] is not [Dale]\'s mother\"\n" + \
            "Answer:\nLet\'s think step by step. \n1. [Dale] is the father of [Charlotte].\n2. [Ashley] is the grandmother of [Charlotte]. \n3. If [Dale] is father of [Charlotte] and [Ashley] is the grandmother of [Charlotte] then [Dale] is the son of [Ashley].\n4. If [Dale] is the son of [Ashley], then [Ashley] is the mother of [Dale]. " + \
            "\nTherefore, the answer to the question is No, the statement is ot true.\n***\n"

In [ ]:
import pickle as pkl

In [ ]:
outs_str = '/home/XXXX/XXXX/
outs = pkl.load(open(outs_str, 'rb'))

In [ ]:
len(outs)

In [ ]:
# for key, value in outs.items():
#     print(value[5])

In [ ]:
print()

In [ ]:
len(outs)

In [ ]:
for key, value in labels.items():
    labels[key] = value.strip(' ')

In [ ]:
outs_pred = {}
outs_acc = 0
num_trues = 0
true_pos = 0
false_pos = 0
true_neg = 0
false_neg = 0
n_false = 0
n_true = 0
for key, value in outs.items():
    if value[4] == True:
        if len(value[1]['neg']) == 0 and labels[key] == 'false':
            outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'true':
            outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    else:
        if len(value[1]['neg']) == 0 and labels[key] == 'true':
            outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'false':
            outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    if labels[key] == 'true':
        num_trues += 1
    
    if labels[key] == 'false':
       n_false += 1
    elif labels[key] == 'true':
        n_true += 1
outs_acc /= len(outs_pred.keys())
# outs['clutrr545.cnf'][1]
print(outs_acc)
print(outs_acc*len(outs_pred.keys()))
print(n_true, n_false)

In [ ]:
n_false = 0
n_true = 0
for key, value in labels.items():
    if labels[key] == 'false':
       n_false += 1
    elif labels[key] == 'true':
        n_true += 1

In [ ]:
name_idx = {}
i = 0
for name in names:
    name_idx[name] = i
    i += 1

In [ ]:
for key, value in labels.items():
    labels[key] = value.strip(' ')

In [ ]:
labels[key]

In [ ]:
temp_outs = pkl.load(open(temp_outs_str, 'rb'))


In [ ]:
len(temp_outs)

In [ ]:
# /home/XXXX/XXXX/fs_backup_feb13/all_outs_thresh09_rulethresh04_contexthresh05_dynamicTure.pkl
missed = pkl.load(open('//home/XXXX/XXXX/fs_backup_feb13//missed_list_' + outs_str, 'rb'))
hunh_list = pkl.load(open('/home/XXXX/XXXX/fs_backup_feb13/hunh_' + outs_str, 'rb'))

In [ ]:
temp_outs_str = '/home/XXXX/XXXX/'
temp_outs = pkl.load(open(temp_outs_str, 'rb'))
import torch
temp_outs_pred = {}
outs_acc = 0
num_trues = 0
true_pos = 0
false_pos = 0
true_neg = 0
false_neg = 0
n_false = 0
n_true = 0
for key, value in temp_outs.items():
    if value[4] == True:
        if len(value[1]['neg']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    else:
        if len(value[1]['neg']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    if labels[key] == 'true':
        num_trues += 1
    
    if labels[key] == 'false':
       n_false += 1
    elif labels[key] == 'true':
        n_true += 1
outs_acc /= len(temp_outs_pred.keys())
# outs['clutrr545.cnf'][1]
print(outs_acc)
print(outs_acc*len(temp_outs_pred.keys()))
print(n_true, n_false)

scs_70 = []
flag_70 = []
lens_70 = []
fgf_70 = []
fbf_70 = []
colors = ['r', 'g']
skips = 0
for key in list(temp_outs.keys()):
    mat = torch.stack(temp_outs[key][5]) / torch.stack(temp_outs[key][5]).sum(1).reshape(-1,1)
    mat = mat[:,0]
    if labels[key] == 'true':
        mat = torch.concatenate((torch.tensor([n_votes[name_idx[key]]/20]), mat))
    else:
        mat = torch.concatenate((torch.tensor([1-(n_votes[name_idx[key]]/20)]), mat))
    lens_70.append(len(mat)-1)
    if labels[key] == 'true':
        if mat[0] < 0.5 and mat[-1] > 0.5:
            flag_70.append(2)
            for z in range(len(mat)):
                if mat[z] > 0.5:
                    fgf_70.append(z)
                    break
        elif mat[0] > 0.5 and mat[-1] < 0.5:
            flag_70.append(3)
            for z in range(len(mat)):
                if mat[z] < 0.5:
                    fbf_70.append(z)
                    break
        else: flag_70.append(temp_outs_pred[key])
    else:
        if mat[0] > 0.5 and mat[-1] < 0.5:
            flag_70.append(2)
            for z in range(len(mat)):
                if mat[z] < 0.5:
                    fgf_70.append(z)
                    break
        elif mat[0] < 0.5 and mat[-1] > 0.5:
            flag_70.append(3)
            for z in range(len(mat)):
                if mat[z] > 0.5:
                    fbf_70.append(z)
                    break
        else: flag_70.append(temp_outs_pred[key])
    # try: flag_70.append(temp_outs_pred[key])
    # except: 
    #     skips += 1
    #     continue
    scs_70.append(mat.clone())


temp_outs_pred = {}
outs_acc = 0
num_trues = 0
true_pos = 0
false_pos = 0
true_neg = 0
false_neg = 0
n_false = 0
n_true = 0
for key, value in temp_outs.items():
    if value[4] == True:
        if len(value[1]['neg']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    else:
        if len(value[1]['neg']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    if labels[key] == 'true':
        num_trues += 1
    
    if labels[key] == 'false':
       n_false += 1
    elif labels[key] == 'true':
        n_true += 1
outs_acc /= len(temp_outs_pred.keys())
# outs['clutrr545.cnf'][1]
print(outs_acc)
print(outs_acc*len(temp_outs_pred.keys()))
print(n_true, n_false)

scs_8 = []
flag_8 = []
lens_8 = []
first_good_flip_8= []
first_bad_flip_8 = []
colors = ['r', 'g', 'b', 'orange']
plot_labels = ['unflipped-wrong', 'unflipped-correct', 'flipped correct', 'flipped incorrect']
skips = 0
for key in list(temp_outs.keys())[:100]:
    mat = torch.stack(temp_outs[key][5]) / torch.stack(temp_outs[key][5]).sum(1).reshape(-1,1)
    mat = mat[:,0]
    if labels[key] == 'true':
        mat = torch.concatenate((torch.tensor([n_votes[name_idx[key]]/20]), mat))
    else:
        mat = torch.concatenate((torch.tensor([1-(n_votes[name_idx[key]]/20)]), mat))
    lens_8.append(len(mat)-1)
    if labels[key] == 'true':
        if mat[0] < 0.5 and mat[-1] > 0.5:
            flag_8.append(2)
            for z in range(len(mat)):
                if mat[z] > 0.5:
                    first_good_flip_8.append(z)
                    break
        elif mat[0] > 0.5 and mat[-1] < 0.5:
            flag_8.append(3)
            for z in range(len(mat)):
                if mat[z] < 0.5:
                    first_bad_flip_8.append(z)
                    break
        else: flag_8.append(temp_outs_pred[key])
    else:
        if mat[0] > 0.5 and mat[-1] < 0.5:
            flag_8.append(2)
            for z in range(len(mat)):
                if mat[z] < 0.5:
                    first_good_flip_8.append(z)
                    break
        elif mat[0] < 0.5 and mat[-1] > 0.5:
            flag_8.append(3)
            for z in range(len(mat)):
                if mat[z] > 0.5:
                    first_bad_flip_8.append(z)
                    break
        else: flag_8.append(temp_outs_pred[key])
    
    scs_8.append(mat.clone())
    
import matplotlib.patches as mpatches
 

fig1, ax1 = plt.subplots()
for i in range(len(lens_8)):
    for j in range(lens_8[i]):
        lens_8.append(j)
for i in range(len(scs_8)):
    ax1.plot(scs_8[i], c=colors[int(flag_8[i])],label=plot_labels[int(flag_8[i])])
line = [1]
for i in range(0,6):
    line.append(1-i*0.1)
line2 = [0]
for i in range(0,6):
    line2.append(i*0.1)
ax1.plot(line,'--', c='black')
ax1.plot(line2,'--', c='black')

ax1.set_title('8B \"True\" classification confidence over iterations (n=' + str(len(scs_8))+')')
# fig5, ax5 = plt.subplots()

ax1.hist(lens_8, density=True, alpha=0.4, bins=range(8))
# flag_8.append(2)

# un, flag8_counts = np.unique(flag_8, return_counts=True)
# patches = []
# for i in range(len(colors)):
#     patches.append(mpatches.Patch(color=colors[i], label=plot_labels[i] + ' (n=' + str(flag8_counts[i]) + ')'))
# ax1.legend(handles=patches)
# # ax1.set_xticks(list(range(0,11)))
# # ax1.set_xticklabels(list(range(1,12)))


for i in range(len(lens_70)):
    for j in range(lens_70[i]):
        lens_70.append(j)
fig2, ax2 = plt.subplots()
for i in range(len(scs_70)):
    ax2.plot(scs_70[i], c=colors[int(flag_70[i])], label=plot_labels[int(flag_70[i])])
ax2.set_title('70B \"True\" classification confidence over iterations (n=' + str(len(scs_70))+')')
# flag_70.append(3)
un, flag70_counts = np.unique(flag_70, return_counts=True)
patches = []
for i in range(len(colors)):
    patches.append(mpatches.Patch(color=colors[i], label=plot_labels[i] + ' (n=' + str(flag70_counts[i]) + ')'))
ax2.legend(handles=patches)
ax2.hist(lens_70, density=True, alpha=0.4, bins=range(8))
ax2.plot(line,'--', c='black')
ax2.plot(line2,'--', c='black')
# ax2.set_xlim(0,6) 

# ax2.set_xticks(list(range(0,11)))
# ax2.set_xticklabels(list(range(1,12)))

fig3, ax3 = plt.subplots()
ax3.hist(first_good_flip_8, label='First Good Flip Iteration')
ax3.set_title("8B - Iteration of Good Flip")
fig4, ax4 = plt.subplots()
ax4.hist(first_bad_flip_8, label='First Bad Flip Iteration')
ax4.set_title("8B - Iteration of Bad Flip")

fig5, ax5 = plt.subplots()
ax5.hist(fgf_70, label='First Good Flip Iteration')
ax5.set_title("70B - Iteration of Good Flip")
fig6, ax6 = plt.subplots()
ax6.hist(fbf_70, label='First Bad Flip Iteration')
ax6.set_title("70B - Iteration of Bad Flip")
# ax3.legend()

In [ ]:
flag70_counts

In [ ]:
x = []
fx = [[],[],[],[]]
fy = [[],[],[],[]]
y = []
for j in range(len(scs_70)):
    s = scs_70[j]
    for i in range(len(s)):
        x.append(i)
        y.append(s[i])
        fy[flag_70[j]].append(s[i])
        fx[flag_70[j]].append(i)


In [ ]:
y_a = np.array(y)

In [ ]:
y_a.max()

In [ ]:
hist, xedges, yedges = np.histogram2d(x, y, bins=[[2,3,4,5,6,7],[0,0.2,0.4,0.6,0.8]])


#  range=[np.arange(0, 8, 1), np.arange(0, 1, 0.2)]


fig = plt.figure()
ax = fig.add_subplot(projection='3d')


# Construct arrays for the anchor positions of the 16 bars.
xpos, ypos = np.meshgrid(xedges[:-1], yedges[:-1], indexing="ij")
xpos = xpos.ravel()
ypos = ypos.ravel()
zpos = 0

# Construct arrays with the dimensions for the 16 bars.
dx = dy = 0.5 * np.ones_like(zpos)
dz = hist.ravel()

ax.bar3d(xpos, ypos, zpos, 0.5, 0.2, dz, zsort='average')

In [ ]:
from matplotlib.colors import LightSource
fhist = []
for i in range(4):
    hist, xedges, yedges = np.histogram2d(fx[i], fy[i], bins=[[2,3,4,5,6,7],[0,0.2,0.4,0.6,0.8,1]])
    fhist.append(hist)
    # xedges.append
    
# xedges *= 100
colors = ['r', 'g', 'b', 'orange']

#  range=[np.arange(0, 8, 1), np.arange(0, 1, 0.2)]


fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.view_init(elev=40, azim=320, roll=0)

plot_labels = ['unflipped-wrong', 'unflipped-correct', 'flipped correct', 'flipped incorrect']

# Construct arrays for the anchor positions of the 16 bars.
xpos, ypos = np.meshgrid(xedges[:-1], yedges[:-1], indexing="ij")
xpos = xpos.ravel()
ypos = ypos.ravel()*100
zpos = 0

# Construct arrays with the dimensions for the 16 bars.
dx = dy = 0.5 * np.ones_like(zpos)
dz = fhist[0].ravel()

# ax.bar3d(xpos, ypos, zpos, 0.5,0.5, dz, zsort='max', color = colors[0])
cumhist = np.zeros_like(dz)
for j in range(len(xpos)):
    x = xpos[j]
    y = ypos[j]
    cumhist=0
    # print(x,y)
    for i in [3,0,2,1]:

        # fig = plt.figure()
        # ax = fig.add_subplot(projection='3d')
        dz = fhist[i].ravel()[j]
        # dz = fhist[i].ravel()
        ax.bar3d(x, y, cumhist, 0.5, 10,dz ,zorder=0, color=colors[i], lightsource=LightSource(azdeg=190))
        cumhist += dz

ax.set_xlabel('Iteration Number')
ax.set_ylabel('confidence')
ax.set_title('Histogram of Confidences as ARGOS Iterates')
ax.set_yticklabels([str(i) + '%' for i in [-100, -60, -20, 20, 60, 100]])
ax.set_ylim(0, 101)
patches = []
for i in range(len(colors)):
    patches.append(mpatches.Patch(color=colors[i], label=plot_labels[i] + ' (n=' + str(flag70_counts[i]) + ')'))
ax.legend(handles=patches,bbox_to_anchor=(1.3,1))
# ax.legend(handles=patches,bbox_to_anchor=(0.7,-0.1))
fig.savefig('./threedhist.pdf', bbox_inches='tight')


In [ ]:
lvsf = [[],[],[],[]]
totals = []
rebins = [0, 0.2, 0.4, 0.6, 0.8, 1]
for i in range(len(scs_70)):
    b = np.digitize(scs_70[i][0], bins)
    # print(b)
    lvsf[flag_70[i]].append(len(scs_70[i])-1)
    totals.append(len(scs_70[i])-1)
    
fig, ax = plt.subplots()
ax.hist(lvsf, stacked=True, color=cs)
patches = []
# ax.legend()
for i in range(len(cs)):
    patches.append(mpatches.Patch(color=cs[i], label=plot_labels[i] + ' (n=' + str(flag70_counts[i]) + ')'))
ax.legend(handles=patches,bbox_to_anchor=(1.3,1))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example data for 3 groups
np.random.seed(0)
# data1 = np.random.normal(0, 1, 1000)
# data2 = np.random.normal(1, 1, 1000)
# data3 = np.random.normal(2, 1, 1000)
data0 = lvsf[0]
data1 = lvsf[1]
data2 = lvsf[2]
data3 = lvsf[3]
# Define bin edges
# bins = np.linspace(-4, 6, 15)
bins = np.array([1,2,3,4,5,6,7])

# Get histogram counts for each dataset
counts1, _ = np.histogram(data1, bins=bins)
counts2, _ = np.histogram(data2, bins=bins)
counts3, _ = np.histogram(data3, bins=bins)
counts0, _ = np.histogram(data0, bins=bins)

# Stack the counts
counts = np.vstack([counts0, counts1, counts2, counts3])

# Normalize so each column (bin) sums to 1
normalized_counts = counts / counts.sum(axis=0, keepdims=True)

# Handle divide-by-zero (empty bins)
normalized_counts = np.nan_to_num(normalized_counts)

# Plot the stacked histogram with normalized heights
bin_centers = 0.5 * (bins[:-1] + bins[1:])
width = np.diff(bins)

# Plot
fig, ax = plt.subplots()
bottom = np.zeros_like(bin_centers)

# colors = ['blue', 'orange', 'green']
# labels = ['Data1', 'Data2', 'Data3']

for i in range(normalized_counts.shape[0]):
    ax.bar(bin_centers, normalized_counts[i], width=width, bottom=bottom,
           color=cs[i], label=plot_labels[i], edgecolor='black')
    bottom += normalized_counts[i]

ax.set_ylabel('Proportion')
ax.set_xlabel('# ARGOS Iterations Before Exit')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.set_yticks([])
ax.set_xticks([1,2,3,4,5,6])
ax.set_xticklabels([1,2,3,4,5,6])
____, totalcounts = np.unique(totals, return_counts=True)

ax_top = fig.add_axes([0.125, 0.85, 0.775, 0.2])  # [left, bottom, width, height]
ax_top.hist(totals, bins=[1,2,3,4,5,6,7], alpha=0.6)
# ax_top.set_xlabel('Iteration Count')
ax_top.spines['top'].set_visible(False)
ax_top.spines['right'].set_visible(False)
ax_top.spines['bottom'].set_visible(False)
ax_top.spines['left'].set_visible(False)
ax_top.set_yticks([])
ax_top.set_xticks([])
ax_top.set_ylabel('Total count')
for c in range(len(totalcounts)):
    ax_top.text(x = 1.2 + 0.05 + 1*c, y = totalcounts[c] + 100, s = str(totalcounts[c]))
# ax.set_title('Normalized Stacked Histogram')
# ax.legend()
patches = []
for i in range(len(cs)):
    patches.append(mpatches.Patch(color=cs[i], label=plot_labels[i] + ' (n=' + str(flag70_counts[i]) + ')'))
ax_top.legend(handles=patches,bbox_to_anchor=(0.3,0.5))
# plt.show()
fig.savefig('./lenhist.pdf', bbox_inches='tight')

In [ ]:
____, totalcounts = np.unique(totals, return_counts=True)


In [ ]:
totalcounts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example data
np.random.seed(0)
data1 = np.random.normal(0, 1, 1000)
data2 = np.random.normal(1, 1, 1000)
data3 = np.random.normal(2, 1, 1000)

# Bin setup
bins = np.linspace(-4, 6, 15)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
width = np.diff(bins)

# Hist counts
counts1, _ = np.histogram(data1, bins=bins)
counts2, _ = np.histogram(data2, bins=bins)
counts3, _ = np.histogram(data3, bins=bins)

# Stack counts
counts = np.vstack([counts1, counts2, counts3])
total_counts = counts.sum(axis=0)

# Normalize
normalized_counts = counts / counts.sum(axis=0, keepdims=True)
normalized_counts = np.nan_to_num(normalized_counts)

# Create main and top axes
fig, ax = plt.subplots(figsize=(10, 6))
divider_height = 0.25
ax_top = fig.add_axes([0.125, 0.75, 0.775, 0.2], sharex=ax)  # [left, bottom, width, height]

# --- Top histogram (total counts) ---
ax_top.bar(bin_centers, total_counts, width=width, color='lightgray', edgecolor='black')
# ax_top.set_ylabel("Total count")
ax_top.spines['bottom'].set_visible(False)
ax_top.tick_params(labelbottom=False)  # Hide x-ticks for top plot
ax_top.set_yticks([])
ax_top.spines['top'].set_visible(False)
ax_top.spines['right'].set_visible(False)
ax_top.spines['bottom'].set_visible(False)
ax_top.spines['left'].set_visible(False)

# --- Bottom normalized stacked bar plot ---
bottom = np.zeros_like(bin_centers)
colors = ['blue', 'orange', 'green']
labels = ['Data1', 'Data2', 'Data3']

for i in range(normalized_counts.shape[0]):
    ax.bar(bin_centers, normalized_counts[i], width=width, bottom=bottom,
           color=colors[i], edgecolor='black', label=labels[i])
    bottom += normalized_counts[i]

ax.set_ylabel('Proportion (normalized)')
ax.set_xlabel('Value')
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
np.array(lvsf[4]).max()

In [ ]:
np.digitize(scs_70[0], bins)
print(flag_70[0])

In [ ]:
bins = [0.01, 0.25, 0.5, 0.75, 1.1]
rebins = [0, 0.2, 0.4,0.6, 0.8, 1]
bins = np.array(rebins)+0.01
fvsc = [[],[],[],[]]
for i in range(len(flag_70)):
    fvsc[flag_70[i]].append(rebins[np.digitize(scs_70[i][0], bins)])

In [ ]:
fvsc[1][0]

In [ ]:
from matplotlib import colors
print(colors.ListedColormap('Accent').colors)
cs = ['r', 'g', 'b', 'orange']
plot_labels = ['unflipped-wrong', 'unflipped-correct', 'flipped correct', 'flipped incorrect']
import matplotlib as mpl

n_lines = 8
cmap = mpl.colormaps['Accent']

# Take colors at regular intervals spanning the colormap.
cs = cmap(np.linspace(0, 1, n_lines))
cs = [cs[5], cs[0], cs[1], cs[2]]
print(cs)

In [ ]:
fig, ax = plt.subplots()
ax.hist(fvsc, stacked=True, bins=rebins, color=cs)
patches = []
# ax.legend()
for i in range(len(cs)):
    patches.append(mpatches.Patch(color=cs[i], label=plot_labels[i] + ' (n=' + str(flag70_counts[i]) + ')'))
ax.legend(bbox_to_anchor=(0.15, 1.33), loc=2, borderaxespad=0, handles=patches)
ax.set_xticks(rebins)
# ax.set_xticklabels(['-100%', '-50%', '0%', '50%', '100%'])
ax.set_yticks([])
# ax.set_yticklabel([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.set_xlabel('Initial Solvability (Confidence)')
fig.savefig('./2dhist.pdf', bbox_inches='tight')

In [ ]:
padded_scs_70 = []

from matplotlib import colors
for i in range(len(scs_70))[100:150]:
    # padded_scs_70.append(list(scs_70[i]))
    for z in range(2):
        padded_scs_70.append([])
        for j in range(len(scs_70[i])):
            padded_scs_70[-1] += [(scs_70[i][j])]*10
        for j in range(len(scs_70[i]), 15):
            # padded_scs_70[-1] += [torch.tensor(-1000000000000000000)]*100

            padded_scs_70[-1] += [torch.tensor(-1)]*10


        


In [ ]:
padded_scs_70_sorted = sorted(padded_scs_70, key= lambda x: x[0])
final = []
for s in padded_scs_70_sorted:
    for z in range(len(s)):
        if s[z] == -1:
            final.append(s[z-1])
            break
final = np.array(final)
final_as = final.argsort()
padded_scs_70_sorted = [padded_scs_70_sorted[i] for i in final_as]

In [ ]:
import operator

scs_70_sorted = sorted(scs_70[100:150], key=operator.itemgetter(0, -1))

In [ ]:
padded_scs_70_sorted = []
scs_70_sorted = sorted(scs_70[100:150], key = lambda x: x[0])
for i in range(len(scs_70_sorted)):
    for z in range(2):
        padded_scs_70_sorted.append([])

        for j in range(len(scs_70_sorted[i])):
            padded_scs_70_sorted[-1] += [(scs_70_sorted[i][j])]*10
        for j in range(len(scs_70_sorted[i]), 15):
            padded_scs_70_sorted[-1] += [torch.tensor(-1)]*10

In [ ]:
bins = [-0.01, 0, 0.2, 0.4, 0.6, 0.8, 1, 1.02]
# scs_70_sorted = sorted(scs_70[100:150], key=operator.itemgetter(0, -1))
scs_70_sorted = scs_70[100:150]

d_scs_s = []
for i in range(len(scs_70_sorted)):
    d_scs_s.append(np.digitize(scs_70_sorted[i], bins))
r_scs_s = []
for i in range(len(d_scs_s)):
    r_scs_s.append([])
    for j in range(len(d_scs_s[i])):
        r_scs_s[-1].append(bins[d_scs_s[i][j]-1])
                   

In [ ]:
padded_scs_70_sorted = []
# scs_70_sorted = sorted(scs_70, key = lambda x: x[0])
scs_70_sorted = sorted(r_scs_s, key=operator.itemgetter(0,1, -1))

from matplotlib import colors
for i in range(len(scs_70_sorted)):
    # padded_scs_70.append(list(scs_70[i]))
    for z in range(2):
        padded_scs_70_sorted.append([])
        for j in range(len(scs_70_sorted[i])):
            padded_scs_70_sorted[-1] += [(scs_70_sorted[i][j])]*10
        for j in range(len(scs_70_sorted[i]), 15):
            # padded_scs_70[-1] += [torch.tensor(-1000000000000000000)]*100

            padded_scs_70_sorted[-1] += [torch.tensor(-1)]*10


        


In [ ]:
np.stack(padded_scs_70_sorted).max()

In [ ]:
from matplotlib import colors
# bounds = [-1, 0.49,0.59, 0.69, 0.79, 0.89, 0.99, 1.09]
bounds = [-1, -0.01, 0.19, 0.39, 0.59, 0.79, 1.01]
norm = colors.BoundaryNorm(bounds, len(bounds))
cmap = 'Paired'
figs, axs = plt.subplots(1,1)
im = axs.imshow(padded_scs_70_sorted, cmap=cmap, norm=norm)
# im = axs.imshow([[0.1]*10]*10, cmap=cmap, norm=norm)

axs.set_xticks([0,10, 20, 30, 40, 50, 60])
axs.set_yticks([0, 20, 40, 60, 70, 80, 100])
axs.set_xlim(0, 70)
axs.set_xticklabels(['SC',1, 2,3, 4, 5,6])
axs.set_xlabel('Number of Iterations')
axs.set_ylabel('Question ID')
axs.set_yticklabels([0, 10, 20, 30, 35, 40,50])
# axs.set_
# axs.set_xlim(0, 35)
# cbar = figs.colorbar(im, ax=axs, ticks=[-0.25, 0.55, 0.65, 0.75, 0.85, 0.95, 1.05, 1.15])
cbar = figs.colorbar(im, ax=axs)
# cbar.ax.set_yticks([-0.25, 0.55, 0.65, 0.75, 0.85, 0.95, 1.05])
# cbar.ax.set_yticks(np.array([-1, 0, 0.2, 0.4, 0.6, 0.8, 1.0]))
# cbar.set_ylim(-1, 1)

# cbar.ax.set_yticklabels(['terminated'] + [str(i) + '%' for i in list(((np.array(bounds[1:-1])+0.01)*100).astype(int))])

cbar.ax.set_yticklabels(['terminated'] + [str(i) + '%' for i in [-100, -60, -20, 20, 60, 100]] )

# cbar.ax.set_ylabel('Confidence (%)')
# grad = []
# for b in bounds:
#     grad+= [([b+0.01]*10)]*10
# # grad = [gr
# # fig1, ax1 = plt.subplots()
# axs[1].imshow(grad, cmap=cmap, norm=norm, origin='lower')

# axs[1].set_yticklabels(['terminated'] + list(np.array(bounds[1:-1])+0.01))
# axs[1].set_yticks([5, 15, 25, 35, 45, 55, 65, 75])
# # ax1.set_yticklabels(['terminated'
# axs[1].set_xticks([])
# axs[1].set_ylim(0,65)
figs.savefig('./cbarplot.pdf', bbox_inches='tight')

In [ ]:
padded_scs_70 = []

from matplotlib import colors
for i in range(len(scs_70))[100:150]:
    # padded_scs_70.append(list(scs_70[i]))
    for z in range(2):
        padded_scs_70.append([])
        for j in range(len(scs_70[i])):
            padded_scs_70[-1] += [(scs_70[i][j])]*10
        for j in range(len(scs_70[i]), 15):
            # padded_scs_70[-1] += [torch.tensor(-1000000000000000000)]*100

            padded_scs_70[-1] += [torch.tensor(-1)]*10


        


In [ ]:
from matplotlib import colors
# bounds = [-1, 0.49,0.59, 0.69, 0.79, 0.89, 0.99, 1.09]
bounds = [-1, -0.01, 0.19, 0.39, 0.59, 0.79, 0.99, 1.19]
norm = colors.BoundaryNorm(bounds, len(bounds))
cmap = 'Paired'
figs, axs = plt.subplots(1,1)
im = axs.imshow(padded_scs_70, cmap=cmap, norm=norm)
axs.set_xticks([0,10, 20, 30, 40, 50, 60])
axs.set_yticks([0, 20, 40, 60, 68, 80, 100])
axs.set_xlim(0, 70)
axs.set_xticklabels(['SC',1, 2,3, 4, 5,6])
axs.set_xlabel('Number of Iterations')
axs.set_ylabel('Question ID')
axs.set_yticklabels([0, 10, 20, 30, 34, 40,50])
# axs.set_
# axs.set_xlim(10, 60)
# cbar = figs.colorbar(im, ax=axs, ticks=[-0.25, 0.55, 0.65, 0.75, 0.85, 0.95, 1.05, 1.15])
cbar = figs.colorbar(im, ax=axs)
# cbar.ax.set_yticks([-0.25, 0.55, 0.65, 0.75, 0.85, 0.95, 1.05])
cbar.ax.set_yticks([-0.5, 0.1, 0.3, 0.5, 0.7, 0.9, 1.1])

cbar.ax.set_yticklabels(['terminated'] + [str(i) + '%' for i in list(((np.array(bounds[1:-1])+0.01)*100).astype(int))])

cbar.ax.set_yticklabels(['terminated'] + [str(i) + '%' for i in [-100, -60, -20, 20, 60, 100]])

# cbar.ax.set_ylabel('Confidence (%)')
# grad = []
# for b in bounds:
#     grad+= [([b+0.01]*10)]*10
# # grad = [gr
# # fig1, ax1 = plt.subplots()
# axs[1].imshow(grad, cmap=cmap, norm=norm, origin='lower')

# axs[1].set_yticklabels(['terminated'] + list(np.array(bounds[1:-1])+0.01))
# axs[1].set_yticks([5, 15, 25, 35, 45, 55, 65, 75])
# # ax1.set_yticklabels(['terminated'
# axs[1].set_xticks([])
# axs[1].set_ylim(0,65)
figs.savefig('./cbarplot.pdf', bbox_inches='tight')

In [ ]:
temp_outs[list(temp_outs.keys())[147]]

In [ ]:
print(list(temp_outs.keys())[147])


In [ ]:
data[int(list(temp_outs.keys())[102].split('clutrr')[1].split(".")[0])]

In [ ]:
print()

In [ ]:
print(list(temp_outs.keys())[118])


In [ ]:
data[int(list(temp_outs.keys())[119].split('clutrr')[1].split(".")[0])]

In [ ]:
print(list(temp_outs.keys())[119])
print(temp_outs[list(temp_outs.keys())[119]])

In [ ]:
list(temp_outs.keys())[119]

In [ ]:
scs_70[119]

In [ ]:
from matplotlib.colors import LightSource
fhist = []
for i in range(4):
    hist, xedges, yedges = np.histogram2d(fx[i], fy[i], bins=[[2,3,4,5,6,7],[0,0.2,0.4,0.6,0.8,1]])
    fhist.append(hist)
    # xedges.append


#  range=[np.arange(0, 8, 1), np.arange(0, 1, 0.2)]

colors = ['r', 'g', 'g', 'r']
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.view_init(elev=40, azim=320, roll=0)


# Construct arrays for the anchor positions of the 16 bars.
xpos, ypos = np.meshgrid(xedges[:-1], yedges[:-1], indexing="ij")
xpos = xpos.ravel()
ypos = ypos.ravel()
zpos = 0

# Construct arrays with the dimensions for the 16 bars.
dx = dy = 0.5 * np.ones_like(zpos)
dz = fhist[0].ravel()

# ax.bar3d(xpos, ypos, zpos, 0.5,0.5, dz, zsort='max', color = colors[0])
cumhist = np.zeros_like(dz)
for j in range(len(xpos)):
    x = xpos[j]
    y = ypos[j]
    cumhist=0
    # print(x,y)
    for i in [3,0,2,1]:

        # fig = plt.figure()
        # ax = fig.add_subplot(projection='3d')
        dz = fhist[i].ravel()[j]
        # dz = fhist[i].ravel()
        ax.bar3d(x, y, cumhist, 0.5, 0.1,dz ,zorder=0, color=colors[i], lightsource=LightSource(azdeg=180))
        cumhist += dz

ax.set_xlabel('Iteration Number')
ax.set_ylabel('Confidence')
ax.set_title('Histogram of Confidences as ARGOS Iterates')
patches = []
colors = ['r', 'g']
plot_labels = ['incorrect', 'correct']
simple_flag70_counts = [flag70_counts[0]+ flag70_counts[3], flag70_counts[1] + flag70_counts[2]]
for i in range(2):
    patches.append(mpatches.Patch(color=colors[i], label=plot_labels[i] + ' (n=' + str(simple_flag70_counts[i]) + ')'))
ax.legend(handles=patches,bbox_to_anchor=(1.2,1))
# ax.legend(handles=patches,bbox_to_anchor=(0.7,-0.1))
fig.savefig('./threedhist_simple.pdf')


In [ ]:
xpos, ypos

In [ ]:
len(ypos)

In [ ]:
colors

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create sample data
data1 = np.random.normal(0, 1, 100)
data2 = np.random.normal(2, 1, 100)
data3 = np.random.normal(4, 1, 100)

# Define bins
bins = np.linspace(-5, 10, 20)

# Create the histogram data
hist_data1, _ = np.histogram(data1, bins=bins)
hist_data2, _ = np.histogram(data2, bins=bins)
hist_data3, _ = np.histogram(data3, bins=bins)

# Set up the figure and axes for 3D plotting
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Create the x coordinates for the bars
x = np.arange(len(hist_data1))

# Define the width of the bars
width = 0.2

# Plot the stacked bars
ax.bar3d(x, np.zeros(len(hist_data1)), np.zeros(len(hist_data1)), width, hist_data1, 0.5, shade=True, label='Data 1')
ax.bar3d(x + width, np.zeros(len(hist_data2)), hist_data1, width, hist_data2, 0.5, shade=True, label='Data 2')
ax.bar3d(x + 2 * width, np.zeros(len(hist_data3)), hist_data1 + hist_data2, width, hist_data3, 0.5, shade=True, label='Data 3')

# Set labels and title
ax.set_xlabel('Bins')
ax.set_ylabel('')
ax.set_zlabel('Frequency')
ax.set_xticks(x + width, bins[:-1], rotation=45, ha='right')
ax.set_title('Stacked 3D Histogram')

# Add legend
# ax.legend()

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure()
X = np.stack([np.ones(4)*i for i in range(5)])
Y = np.stack([np.ones(5)*i for i in np.arange(0, 0.8, 0.2)]).transpose()
# X,Y = np.meshgrid(xpos, ypos)
ax = plt.axes(projection ='3d')
ax.plot_wireframe(X, Y, hist, color ='green')
ax.set_title('wireframe geeks for geeks');

In [ ]:
temp_outs_str = 'home/XXXX/XXXX/'
temp_outs = pkl.load(open(temp_outs_str, 'rb'))
import torch
temp_outs_pred = {}
outs_acc = 0
num_trues = 0
true_pos = 0
false_pos = 0
true_neg = 0
false_neg = 0
n_false = 0
n_true = 0
for key, value in temp_outs.items():
    # if len(value[5]) <= 3: continue
    if value[4] == True:
        if len(value[1]['neg']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    else:
        if len(value[1]['neg']) == 0 and labels[key] == 'true':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_neg += 1
        elif len(value[1]['pos']) == 0 and labels[key] == 'false':
            temp_outs_pred[key] = True
            outs_acc += 1
            true_pos += 1
        else:
            temp_outs_pred[key] = False
            if labels[key] == 'true':
                false_neg += 1
            else:
                false_pos += 1
    if labels[key] == 'true':
        num_trues += 1
    
    if labels[key] == 'false':
       n_false += 1
    elif labels[key] == 'true':
        n_true += 1
print(true_pos, true_neg, false_pos, false_neg)

In [ ]:
len(temp_outs)

In [ ]:
int(temp_outs_pred[key])

In [ ]:
value

In [ ]:
len(temp_outs)

In [ ]:
torch.stack(outs[key][5]).sum(0)

In [ ]:
# for key in outs.keys():
#     print(torch.stack(outs[key][5]))
#     print(torch.stack(outs[key][5]).sum(1))
#     print((torch.stack(outs[key][5]) / torch.stack(outs[key][5]).sum(1).reshape(-1,1)).max(-1))
#     print((torch.stack(outs[key][5]) / torch.stack(outs[key][5]).sum(1).reshape(-1,1)))

In [ ]:
hunh = []
missed_list = []
for miss in missed:
    missed_list.append(miss[0])
for hunh in hunh_list:
    missed_list.append(hunh)

In [ ]:
for key, value in outs.items():

In [ ]:
len(missed_list)

In [ ]:
miss_acc = 0
miss_sc_acc = 0
no_gains = 0
for miss in missed_list:
    if outs_pred[miss] == True:
        miss_acc += 1
    # print(outs_pred[miss])
    if sc_pred[miss] == True:
        miss_sc_acc += 1
    if sc_pred[miss] == False and sc_pred[miss] == False:
        no_gains += 1
    if sc_pred[miss] == True and outs_pred[miss] == True:
        no_gains += 1
print(miss_acc/len(missed_list))
print(miss_sc_acc/len(missed_list))
# print(no_gains/len(missed_list))
print(miss_sc_acc - miss_acc)

In [ ]:
len(missed_list)

In [ ]:
tt = []
tf = []
ft = []
ff = []
for miss in missed_list:
    if sc_pred[miss] == True and outs_pred[miss] == True:
        tt.append(miss)
    elif sc_pred[miss] == True and outs_pred[miss] == False:
        tf.append(miss)
    elif sc_pred[miss] == False and outs_pred[miss] == True:
        ft.append(miss)
    elif sc_pred[miss] == False and outs_pred[miss] == False:
        ff.append(miss)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
tt = []
tf = []
ft = []
ff = []
for name in labels.keys():
    # if name in missed_list:continue
    
    if sc_pred[name] == True and outs_pred[name] == True:
        tt.append(name)
    elif sc_pred[name] == True and outs_pred[name] == False:
        tf.append(name)
    elif sc_pred[name] == False and outs_pred[name] == True:
        ft.append(name)
    elif sc_pred[name] == False and outs_pred[name] == False:
        ff.append(name)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
tt = []
tf = []
ft = []
ff = []
for name in labels.keys():
    if name in missed_list:continue
    
    if cot_pred[0][name] == True and outs_pred[name] == True:
        tt.append(name)
    elif cot_pred[0][name] == True and outs_pred[name] == False:
        tf.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == True:
        ft.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == False:
        ff.append(name)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
tt = []
tf = []
ft = []
ff = []
for name in missed_list:
    # if name in missed_list:continue
    
    if cot_pred[0][name] == True and outs_pred[name] == True:
        tt.append(name)
    elif cot_pred[0][name] == True and outs_pred[name] == False:
        tf.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == True:
        ft.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == False:
        ff.append(name)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
print(tf)

In [ ]:
print(missed_list)

In [ ]:
scores = pkl.load(open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/scores_temp1_thresh075_thresh05_dynFalse_fixed.pkl', 'rb'))

In [ ]:
print(tf)

In [ ]:
i = 1
for line in outs['clutrr60.cnf'][0]:
    print(line)
    if i % 4 == 3:
        # try:
            # print(line.split('known predicate: ')[1].split('. Known predicates are')[0].replace('___', line.split('\\box{ ')[1]))
        print(scores[line.split('known predicate: ')[1].split('. Known predicates are')[0].replace('___', line.split('\\box{ ')[1])])
        # except:
        #     print(line.split('known predicate: ')[1].split('. Known predicates are')[0].replace('___', line.split('\\box{ ')[1]))
            # print(line)
        #     break
    if i%4 == 0 and not str(line).startswith('calls'):
        # continue
        i = 2
        # print('hihi')

    else: i += 1
# print(outs['clutrr125.cnf'])
# print(outs.keys())

In [ ]:
print(list(scores.keys())[0])

In [ ]:
tt = []
tf = []
ft = []
ff = []
for miss in missed_list:
    if cot_pred[0][miss] == True and outs_pred[miss] == True:
        tt.append(miss)
    elif cot_pred[0][miss] == True and outs_pred[miss] == False:
        tf.append(miss)
    elif cot_pred[0][miss] == False and outs_pred[miss] == True:
        ft.append(miss)
    elif cot_pred[0][miss] == False and outs_pred[miss] == False:
        ff.append(miss)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
import torch
score_list = []
for score in scores.values():
    score_list.append(torch.stack(score))
score = torch.stack(score_list)

In [ ]:
score

In [ ]:
 

fig1, ax1 = plt.subplots()
ax1.scatter(x=score[:,0], y=score[:,1], s=3)
ax1.set_xlabel('1 - Does the following rule seem contradictory?')
ax1.set_ylabel('Does the following rule seem contextually relevant?')

In [ ]:
outs_acc*60

In [ ]:
sc_acc

In [ ]:
for key, value in outs_pred.items():
    if key not in missed_list and value == False:
        print(key)

In [ ]:
for i in range(outs['clutrr366.cnf']):
    print(outs['clutrr366.cnf'][i])
# print(outs['clutrr366.cnf'])

In [ ]:
n = 7
for i in range(len(missed[n][1])):
    print(missed[n][1][i])
    # print('\n')

In [ ]:
outs_pred = {}
outs_acc = 0
num_trues = 0
for key, value in outs.items():
    if len(value[1]['neg']) == 0 and labels[key] == 'false':
        outs_pred[key] = True
        outs_acc += 1
    elif len(value[1]['pos']) == 0 and labels[key] == 'true':
        outs_pred[key] = True
        outs_acc += 1
    else:
        outs_pred[key] = False
    if labels[key] == 'true':
        num_trues += 1
outs_acc /= len(outs_pred.keys())
# outs['clutrr545.cnf'][1]

In [ ]:
outs_acc

In [ ]:
outs_pred

In [ ]:
labels[key]

In [ ]:
outs.keys()

In [ ]:
len()

In [ ]:
import shutil

def get_bb(file, del_sols=None):
    bb = {'pos':  [], 'neg': []}
    
    files = ['/'.join(file.split('/')[:-1]) + '/pos_' + file.split('/')[-1], '/'.join(file.split('/')[:-1]) + '/neg_' + file.split('/')[-1] ]
    for i in range(len(files)):
        file = files[i]
        shutil.copy(file, '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
        if not del_sols==None:
            if 'pos' in file:
                if 'neg' in file:
                    print('l. 416 uh oh')
                      
                ds = del_sols['pos']
            elif 'neg' in file:
                ds = del_sols['neg']
            for sol in ds:
                add_clause('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
                cf = open(f'/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]), 'a')
                write_str = '\n'
                for lit in sol:
                    write_str += str(-lit) + ' '
                # write_str += '0'
                cf.write(write_str)
                cf.close()
        # print('running cadical')
        os.system("timeout 5000 /home/XXXX/XXXX/fs_backup_feb13/LLM-project/cadiback/cadiback " + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]) + '> '  + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone")
        #   
        bbone= open('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone", 'r')
        lines = bbone.readlines()
        #   
        for line in lines:
            if line.startswith('b'):
                #   
                lits = line.split(' ')[1:]
                for lit in lits:
                    lit = lit.strip()
                    if lit == '0':
                        continue
                    lit = int(lit)
                    if 'pos' in file:                                
                        if 'neg' in file:
                            print('l. 447 uh oh')
                              
                        bb['pos'].append(lit)
                    elif 'neg' in file:
                            bb['neg'].append(lit)

    return bb


In [ ]:
import shutil

def get_bb(file, del_sols=None):
    bb = {'pos':  [], 'neg': []}
    
    files = ['/'.join(file.split('/')[:-1]) + '/pos_' + file.split('/')[-1], '/'.join(file.split('/')[:-1]) + '/neg_' + file.split('/')[-1] ]
    for i in range(len(files)):
        file = files[i]
        shutil.copy(file, '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
        if not del_sols==None:
            if 'pos' in file:
                if 'neg' in file:
                    print('l. 416 uh oh')
                      
                ds = del_sols['pos']
            elif 'neg' in file:
                ds = del_sols['neg']
            for sol in ds:
                add_clause('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
                cf = open(f'/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]), 'a')
                write_str = '\n'
                for lit in sol:
                    write_str += str(-lit) + ' '
                # write_str += '0'
                cf.write(write_str)
                cf.close()
        # print('running cadical')
        os.system("timeout 5000 /home/XXXX/XXXX/fs_backup_feb13/LLM-project/cadiback/cadiback " + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]) + '> '  + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone")
        #   
        bbone= open('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone", 'r')
        lines = bbone.readlines()
        #   
        for line in lines:
            if line.startswith('b'):
                #   
                lits = line.split(' ')[1:]
                for lit in lits:
                    lit = lit.strip()
                    if lit == '0':
                        continue
                    lit = int(lit)
                    if 'pos' in file:                                
                        if 'neg' in file:
                            print('l. 447 uh oh')
                              
                        bb['pos'].append(lit)
                    elif 'neg' in file:
                            bb['neg'].append(lit)

    return bb

c = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_clutrr_csvs_debug/solver_finished.csv'
import csv
import json
dataset = '/home/XXXX/XXXX/fs_backup_feb13/SAT-LM/data/clutrr_test.json'
with open(dataset, 'r') as df:
    data = json.loads(df.read())

task = 'folio'
missed=False
c = open(c, 'r')
cr = csv.reader(c)
names = []
all_outs = {}
missed_list = []
labels = {}
for row in cr:
    if row[2] == 'SAT' and row[3] == 'SAT':
        cnf = open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_clutrr/neg_'+row[1]).readlines()[0].strip('\n')
        num_clause = int(cnf.split(' ')[-1])
       
        if task=='folio':
            bb = get_bb('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_clutrr/'+row[1])
            jb = set(bb['pos']).intersection(set(bb['neg']))
            if len(jb) == 0:
                continue
        # if num_clause > 500:
            # continue
        names.append(int(row[1].split('clutrr')[1].split('.cnf')[0]))
        labels[row[1]] = data[int(row[1].split('clutrr')[1].split('.')[0])]['label']

In [ ]:
len(names)

In [ ]:
bad_data = []
mistr_data = []
noisy_data=[]
c = '/home/XXXX/LLM-project/dimacs_clutrr_csvs_debug/solver_finished.csv'
import csv
import json
dataset = '/home/XXXX/SAT-LM/data/clutrr_test.json'
with open(dataset, 'r') as df:
    data = json.loads(df.read())
# breakpoint()
task = 'folio'
missed=False
c = open(c, 'r')
cr = csv.reader(c)
names = []
all_outs = {}
missed_list = []
labels = {}
for row in cr:
        if row[2] == 'SAT' and row[3] == 'SAT':
            cnf = open('/home/XXXX/LLM-project/dimacs_clutrr/neg_'+row[1]).readlines()[0].strip('\n')
            num_clause = int(cnf.split(' ')[-1])
            if row[1] in noisy_data or row[1] in mistr_data:
                continue
            if task=='folio':
                bb = get_bb('/home/XXXX/LLM-project/dimacs_clutrr/'+row[1])
                jb = set(bb['pos']).intersection(set(bb['neg']))
                if len(jb) == 0:
                    continue
            # if num_clause > 500:
                # continue
            if row[1] in bad_data:
                continue
            names.append(row[1])
            labels[row[1]] = data[int(row[1].split('clutrr')[1].split('.')[0])]['label']
    #   

In [ ]:
len(names)

In [ ]:
labels

In [ ]:
labels.keys()

In [ ]:
import json
folio = json.load(open('/home/XXXX/SAT-LM/data/lutrr_test.json', 'r'))
folio[48]

In [ ]:
i = 0
cot_acc = 0
cot_preds = {}
for key, value in labels.items():
    if cot[i] == value:
        cot_acc += 1
        cot_preds[key] = True
    else:
        cot_preds[key] = False
    i += 1
print(cot_acc)

In [ ]:
flipped = 0
flipped_names = []
tf = []
ft = []
for name in names:
    if cot_preds['proofd5' + str(name) + '.cnf'] != outs_pred['proofd5' + str(name) + '.cnf']:
        flipped_names.append('proofd5' + str(name) + '.cnf')
        flipped += 1
    if cot_preds['proofd5' + str(name) + '.cnf'] == True and outs_pred['proofd5' + str(name) + '.cnf'] == False:
        tf.append('proofd5' + str(name) + '.cnf')
    if cot_preds['proofd5' + str(name) + '.cnf'] == False and outs_pred['proofd5' + str(name) + '.cnf'] == True:
        ft.append('proofd5' + str(name) + '.cnf')

print(flipped)
print(len(tf))
print(len(ft))

In [ ]:
flipped = 0
flipped_names = []
tf = []
ft = []
for name in missed_list:
    name = name[7:-4]
    if cot_preds['clutrr' + str(name) + '.cnf'] != outs_pred['clutrr' + str(name) + '.cnf']:
        flipped_names.append('clutrr' + str(name) + '.cnf')
        flipped += 1
    if cot_preds['clutrr' + str(name) + '.cnf'] == True and outs_pred['proofd5' + str(name) + '.cnf'] == False:
        tf.append('proofd5' + str(name) + '.cnf')
    if cot_preds['proofd5' + str(name) + '.cnf'] == False and outs_pred['proofd5' + str(name) + '.cnf'] == True:
        ft.append('proofd5' + str(name) + '.cnf')

print(flipped)
print(len(tf))
print(len(ft))

In [ ]:
missed

In [ ]:
print(tf)

In [ ]:
outs['proofd542.cnf']


In [ ]:
name

In [ ]:
ours = pkl.load(open('/home/XXXX/LLM-project/all_outs_temp1_dynFalse.pkl', 'rb'))

In [ ]:
print()

In [ ]:
list(ours.keys())[5]


In [ ]:
cot

In [ ]:
labels[list(ours.keys())[5]]

In [ ]:
outs_str = '/home/XXXX/XXXX/fs_backup_feb13/all_outs_cot_met_clutrr_rulethresh_05_cot_thresh_ANNEALING,_dynamic_False,_sc5_llama_70B,_no_jb_prompt,_fixed_prmopt,_yes_rules_in_prompt,_yes_solver,_shuffled,_old_fewshut,_temp_1,_n_consec_NO-CONSEC,_sc-poo,_COPY_THAT_SAVES_SC_HISTORY.pkl'
outs_70b = pkl.load(open(outs_str, 'rb'))
sc = []
exit_n = []
for key, value in outs_70b.items():
    exit_n.append(len(value[-1]))
    # print(a[-1])
 

plt.hist(exit_n, bins=list(range(10)))

In [ ]:
outs_str = '/home/XXXX/XXXX/fs_backup_feb13/all_outs_cot_met_clutrr_rulethresh_05_cot_thresh_ANNEALING,_dynamic_False,_sc5_llama_8B,_no_jb_prompt,_fixed_prmopt,_yes_rules_in_prompt,_yes_solver,_shuffled,_old_fewshut,_temp_1,_n_consec_NO-CONSEC,_sc-poo,_COPY_THAT_SAVES_SC_HISTORY.pkl'
outs_8b = pkl.load(open(outs_str, 'rb'))
sc = []
exit_n = []
for key, value in outs_8b.items():
    exit_n.append(len(value[-1]))
    # print(a[-1])
 

plt.hist(exit_n, bins=list(range(10)))

In [ ]:
outs_acc_n = {}
outs_totals_n = {}
for i in range(10):
    outs_acc_n[i] = 0
    outs_totals_n[i] = 0
outs_acc = 0
num_trues = 0
# outs_totals_n = {}
for key, value in outs_70b.items():
    outs_totals_n[len(value[-1])] += 1
    if len(value[1]['neg']) == 0 and labels[key].strip(' ') == 'false':
        # outs_pred[key] = True
        outs_acc_n[len(value[-1])] += 1
    elif len(value[1]['pos']) == 0 and labels[key].strip(' ') == 'true':
        # outs_pred[key] = True
        outs_acc_n[len(value[-1])] += 1
    # else:
        # # outs_pred[key] = False
    if labels[key] == 'true':
        num_trues += 1
# for i in range(10):
#     if outs_totals_n[i] == 0: continue
#     outs_acc_n[i] /= outs_totals_n[i]
# outs_acc_n /= len(outs_pred.keys())
# outs['clutrr545.cnf'][1]
# print(outs_acc)
# print(outs_acc*len(outs_pred.keys()))

plt.bar(x = range(10),height=list(outs_acc_n.values()))
plt.title('70B accuracy for iteration length')

In [ ]:
outs_acc_n = {}
outs_totals_n = {}
for i in range(10):
    outs_acc_n[i] = 0
    outs_totals_n[i] = 0
outs_acc = 0
num_trues = 0
# outs_totals_n = {}
for key, value in outs_8b.items():
    outs_totals_n[len(value[-1])] += 1
    if len(value[1]['neg']) == 0 and labels[key].strip(' ') == 'false':
        # outs_pred[key] = True
        outs_acc_n[len(value[-1])] += 1
    elif len(value[1]['pos']) == 0 and labels[key].strip(' ') == 'true':
        # outs_pred[key] = True
        outs_acc_n[len(value[-1])] += 1
    # else:
        # # outs_pred[key] = False
    if labels[key] == 'true':
        num_trues += 1
# for i in range(10):
#     if outs_totals_n[i] == 0: continue
#     outs_acc_n[i] /= outs_totals_n[i]
# outs_acc_n /= len(outs_pred.keys())
# outs['clutrr545.cnf'][1]
# print(outs_acc)
# print(outs_acc*len(outs_pred.keys()))

plt.bar(x = range(10),height=list(outs_acc_n.values()))
plt.title('8B accuracy for iteration length')

In [ ]:
acc = 0
for n, a in outs_acc_n.items():
    acc += outs_totals_n[n]*a
print(acc/len(outs_8b))

In [ ]:
i = 0
sc_acc_n = {}
sc70 = np.where(np.array(n_votes) >= np.ceil(len(cot_pred_list)/2+0.5), 1, 0)
for j in range(10):
    sc_acc_n[j] = 0
for key, value in outs_70b.items():
    if sc70[i] == 1:
        sc_acc_n[len(value[-1])] += 1
    i += 1
# for j in range(10):
#     if outs_totals_n[j] == 0: continue
#     sc_acc_n[j] /= outs_totals_n[j]
ax1, fig1 = plt.subplots()
fig1.bar(x=range(10), height=sc_acc_n.values(), alpha=0.3,label='SC accuracy')
fig1.bar(x=range(10), height=outs_acc_n.values(), alpha=0.3,label='Our Accuracy')


fig1.legend()
# fig1.set_title('SC-20 Llama 70B accuracy on iteration-length divisions, vs SC-5 Llama 8B our-method')
fig1.set_ylabel('Accuracy over the subset')
fig1.set_xlabel('Number of iterations of our method')
fig1.set_yticks(list(np.arange(0,1,0.1)))

In [ ]:
hybrid = outs_acc_n[1] * outs_totals_n[1] + outs_acc_n[2]* outs_totals_n[2] + outs_acc_n[3]*outs_totals_n[3] + \
    sc_acc_n[4]*outs_totals_n[4] + sc_acc_n[5]*outs_totals_n[5] + sc_acc_n[6]*outs_totals_n[6]
print(hybrid/len(outs))


In [ ]:
few_shot = "Facts:\n[Nancy] likes to cut the hair of her daughter [Heidi].\n[Heidi]'s sister [Lorraine] went to beauty school and taught them all how to cut hair expertly. " + \
            "\nHere are some additional facts and rules we\'ve found:\nNancy is the mother of Lorraine\n If [Heidi] is the sister of [Lorraine] and [Heidi] is the daughter of [Nancy] then [Nancy] is the mother of [Lorraine].\n" + \
            "Question: Is the following statement true: \n\"[Lorraine] is [Nancy]\'s daughter\"\n" + \
            "Answer:\nLet\'s think step by step.  \n1. [Heidi] is the sister of [Lorraine]\n2. [Heidi] is the daughter of [Nancy]\n3. If [Heidi] is the sister of [Lorraine] and [Heidi] is the daughter of [Nancy] then [Nancy] is the mother of [Lorraine].\n4. If [Nancy] is the mother of [Lorraine], then [Lorraine] is the daughter of [Nancy].\nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts:\n[Dale] and his sister [Nancy] are decorating for a party.\n[Nancy]'s daughter [Louise] thinks the party will be fun.\n" + \
            "Here are some additional facts and rules we\'ve found:\nDale is the uncle of Louise. If [Nancy] is the sister of [Dale] and [Nancy] is the mother of [Louise] then [Dale] is the uncle of [Louise].\n" + \
            "Question: Is the following statement true: \n\"[Louise] is not [Dales]\'s niece\"\n" + \
            "Answer: Le\'s think step by step. \n1. [Nancy] is the sister of [Dale]. \n2. [Nancy] is the mother of [Louise]\n3.  If [Nancy] is the sister of [Dale] and [Nancy] is the mother of [Louise] then [Dale] is the uncle of [Louise].\n4.If [Dale] is the uncle of [Louise], then [Louise] is the niece of [Dale].\nTherefore, the answer is No, the statement is not true.\n***\n" + \
            "Facts: \n[Lillian] and her sister [Nancy] are the only children in their family. \n[Lillian]'s biggest accomplishment is raising her son [Douglas]. " + \
            "\nHere are some additional facts and rules we\'ve found:\n[Lillian] is the sister of [Nancy]. \nIf [Nancy] is the sister if [Lillian] then [Lillian] is the sister of [Nancy].\n" + \
            "Question: Is the following statement true: \n\"[Douglas] is [Nancy]\'s nephew\"\n" + \
            "Answer:\nLet\'s think step by step. \n1. [Douglas] is [Lillian]\'s son. \n2. [Nancy] is [Lillian]\'s sister. " + \
            "\n3. If [Douglas] is the son of [Lillian] and [Lillian] is the sister of [Nancy] then [Douglas] is the nephew of [Lillian]. \nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts: \n[Ashley] liked to go to the park with her granddaughter [Charlotte]. \n[Dale], [Charlotte]'s father, like to take her to the movies instead. " + \
            "\nHere are some additional facts and rules we\'ve found:\n[Dale] is the son of [Ashley]. If [Dale] is father of [Charlotte] and [Ashley] is the grandmother of [Charlotte] then [Dale] is the son of [Ashley].\n" + \
            "Question: Is the following statement true: \n\"[Ashley] is not [Dale]\'s mother\"\n" + \
            "Answer:\nLet\'s think step by step. \n1. [Dale] is the father of [Charlotte].\n2. [Ashley] is the grandmother of [Charlotte]. \n3. If [Dale] is father of [Charlotte] and [Ashley] is the grandmother of [Charlotte] then [Dale] is the son of [Ashley].\n4. If [Dale] is the son of [Ashley], then [Ashley] is the mother of [Dale]. " + \
            "\nTherefore, the answer to the question is No, the statement is ot true.\n***\n"

print(few_shot)

In [ ]:
import torch, os
from torch.utils.data import DataLoader
os.environ["CURL_CA_BUNDLE"]=""
os.environ["REQUESTS_CA_BUNDLE"]=""
run_log_path = './run_log.txt'
run_log = open(run_log_path, 'w')
run_log.write('hello\n')
run_log.close()
unknown=False
import json
import numpy as np
import csv

In [ ]:
os.environ["CURL_CA_BUNDLE"]=""
os.environ["REQUESTS_CA_BUNDLE"]=""
os.environ["CUDA_VISIBLE_DEVICES"] = '1'
USER_PATH = '/home/XXXX/XXXX/fs_backup_feb13/'
# os.environ['TRANSFORMERS_CACHE'] = '.cache/huggingface/hub'
# cache_dir = '/ephemeral/media/data1/XXXX/hub/'
cache_dir = os.path.join(os.getcwd(), '.cache/huggingface/hub')
os.environ['TRANSFORMERS_CACHE'] = cache_dir
os.environ['HF_HOME'] = cache_dir
# import transformers

# import urllib3
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import argparse
from tqdm import tqdm
import time
import datetime

In [ ]:
import warnings
import contextlib

import requests
from urllib3.exceptions import InsecureRequestWarning

old_merge_environment_settings = requests.Session.merge_environment_settings

@contextlib.contextmanager
def no_ssl_verification():
    opened_adapters = set()

    def merge_environment_settings(self, url, proxies, stream, verify, cert):
        # Verification happens only once per connection so we need to close
        # all the opened adapters once we're done. Otherwise, the effects of
        # verify=False persist beyond the end of this context manager.
        opened_adapters.add(self.get_adapter(url))

        settings = old_merge_environment_settings(self, url, proxies, stream, verify, cert)
        settings['verify'] = False

        return settings

    requests.Session.merge_environment_settings = merge_environment_settings

    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', InsecureRequestWarning)
            yield
    finally:
        requests.Session.merge_environment_settings = old_merge_environment_settings

        for adapter in opened_adapters:
            try:
                adapter.close()
            except:
                pass

In [ ]:
class Struct:
    def __init__(self, **entries):
        self.__dict__.update(entries)

args = {'train_file_path': './example_data', 'test_file_path': './example_data', 'save_path': './../SFT_train_res', 'engine': 'meta-llama/Llama-2-13b-chat-hf', 
        'n_rows': 20, 'max_length': 1000, 'lr': 5e-05, 'weight_decay': 0.0, 'epochs': 10, 'max_grad_norm': 1.0, 'batch_size': 2, 'save_strategy': 'no', 'use_lora': True}
# args['engine'] = 'meta-llama/Meta-Llama-3-70B-Instruct'
# args['engine'] = 'meta-llama/Meta-Llama-3-70B-Instruct'
# args['engine'] = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
args['engine'] = 'Qwen/Qwen2.5-Coder-3B-Instruct'

args = Struct(**args)



In [ ]:

class LLM():
    def __init__(self, args):
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype="bfloat16",
            bnb_4bit_use_double_quant=True,
        )
        with no_ssl_verification():
            

            
            self.tokenizer = AutoTokenizer.from_pretrained(
                    args.engine,
                    cache_dir = cache_dir,
                    token = os.getenv("HF_TOKEN"),
                    attn_implementation="flash_attention_2"

                    )
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id            
            self.model = AutoModelForCausalLM.from_pretrained(
                    args.engine, 
                    cache_dir = cache_dir,
                    quantization_config=quant_config,
                    device_map='auto',
                    token = os.getenv("HF_TOKEN"),
                    attn_implementation="flash_attention_2"
                    )

        self.tokenizer.pad_token = self.tokenizer.eos_token
    
    def sentence_probabilities(self, sentences):
        with torch.no_grad():
            sentence_tokens = self.tokenizer(sentences, return_tensors='pt', padding=True)
            sentence_token_ids = sentence_tokens.input_ids.cuda()

            # Little hack to cut down inference time by 4-5x (leads to some imprecisions when using quantization)
            # Find the common prefix and run it through the model once, to save time
            first_different_token = (sentence_token_ids == sentence_token_ids[0, :].unsqueeze(0)).all(dim=0).long().argmin()
            common_prefix = sentence_token_ids[0, :first_different_token].unsqueeze(0)
            common_prefix_output = self.model(common_prefix, use_cache=True)
            common_prefix_key_values = tuple(tuple(tensor.expand(len(sentences), -1, -1, -1) for tensor in layer) 
                                             for layer in common_prefix_output.past_key_values)

            # Process the rest of the sentences
            rest_outputs = self.model(sentence_token_ids[:, first_different_token:], past_key_values=common_prefix_key_values)
            logits = torch.concat([common_prefix_output.logits.expand(len(sentences), -1, -1), rest_outputs.logits], dim=1).cuda()
            log_probs = logits.log_softmax(-1)
            log_probs = log_probs[:, :-1, :].gather(2, sentence_token_ids[:, 1:][:, :, None]).squeeze(-1).cuda()
            log_probs = (log_probs*sentence_tokens.attention_mask.cuda()[:, 1:]).sum(-1).cpu()
        return log_probs
    def nli(self, sentences, unknown):
        # true_probs = self.sentence_probabilities(sentences + " True.")
        # false_probs = self.sentence_probabilities(sentences + " False.")
        # maybe_probs = self.sentence_probabilities(sentences + " Maybe.")
        if unknown:
            true_probs, maybe_probs, false_probs =  (self.sentence_probabilities([sentences + "(A)", sentences + "(B)", sentences + "(C)"]))
            return {'True': true_probs, 'Maybe': maybe_probs, 'False': false_probs}
        else:
            true_probs, false_probs =  (self.sentence_probabilities([sentences + "(A)", sentences + "(B)"]))
            return {'True': true_probs, 'False': false_probs}
    def yn(self, sentences, norm=True, relaxed=False, obvious=False, fewshot=None, maybe=False):
        yns = []
        for sentence in sentences:
            if fewshot:
                sentence = fewshot + sentence
            
            if relaxed:
                yns.append(sentence + "Most likely")
                yns.append(sentence + "Not necessarily")
            elif obvious:
                yns.append(sentence + "obviously true.")
                yns.append(sentence + "not obviously true.")
            elif maybe:
                yns.append(sentence + "Yes")
                yns.append(sentence + "Maybe")
                yns.append(sentence + "No")
            else:
                yns.append(sentence + "Yes")
                yns.append(sentence + "No")
        # if norm:
        #     norms = self.sentence_probabilities(sentences)
        probs = []
        batch_size = 256
        for i in range(0, len(yns), batch_size):
            if i+batch_size < len(yns):
                probs += list(self.sentence_probabilities(yns[i:i+batch_size]))
            else: 
                probs += list(self.sentence_probabilities(yns[i:]))
        probs=torch.tensor(probs)
        #   
        # probs = (self.sentence_probabilities(yns))
        # probs = torch.exp(probs)
        pyes = []
        pno = []
        pmaybe = []
        if maybe:
            z = 3
        else:
            z = 2
        for i in range(0,len(probs), z):
            # if yns[i] not in cache.keys():
                # yes, no = self.sentence_probabilities([yns[i], yns[i+1]])
            
            if maybe:
                
                yes, maybe, no = probs[i], probs[i+1], probs[i+2]
                
                      
            else:
                yes, no = probs[i], probs[i+1]
            if norm:
                if maybe: 
                    y,m,n = torch.tensor([yes, maybe, no]).softmax(-1)
                else:
                    y,n = torch.tensor([yes, no]).softmax(-1)
              
                # cache[yns[i]] = y
                # cache[yns[i+1]] = n
                pyes.append(y)
                pno.append(n)
                if maybe:
                    pmaybe.append(m)
            else:
                pyes.append(1-yes/(yes + no))
            # else:
            #     y, n = cache[yns[i]], cache[yns[i+1]]
            #     pyes.append(y)
                # pno.append(n)/
        # print('cache length', len(cache))
        # if maybe:
        
        if maybe: return torch.stack([torch.tensor(pyes), torch.tensor(pmaybe), torch.tensor(pno)])
        return torch.tensor(pyes), torch.tensor(pmaybe), torch.tensor(pno)
    def complete(self, prompt, max_new = 25, temp = 1 , topk=0):
        max_length = args.max_length
        encode_ids = self.tokenizer(
        prompt, 
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=len(prompt)+1
    ).input_ids.cuda()
        generated_outputs = self.model.generate(
        encode_ids, 
        max_new_tokens=max_new, 
        return_dict_in_generate=True, 
        output_scores=True,
        temperature=temp,
        top_k=topk
        )
        responses = self.tokenizer.batch_decode(
            generated_outputs.sequences,
            skip_special_tokens=True
        )
        return responses

In [ ]:
llm = LLM(args)

In [ ]:
import pickle as pkl
all_outs = pkl.load(open('/home/XXXX/XXXX/fs_backup_feb13/all_outs_cot_met_clutrr_rulethresh_00_cot_thresh_ANNEALING_05,_sc5,_dynamic_False,_sc5_llama_8B,_no_jb_prompt,_fixed_prmopt,_yes_rules_in_prompt,_yes_solver,_shuffled,_old_fewshut,_temp_1,_n_consec_NO-CONSEC,_sc-poo,_COPY_THAT_SAVES_ALL_COT_PROMPTS.pkl', 'rb'))

In [ ]:
n = 5
preds_70 = {}
preds_8 = {}
answers = ['true', 'false']
acc_8 = 0
acc_70 = 0
n_fewshot = 4
for key, value in all_outs.items():
    votes = torch.tensor([0,0])
    prompt = value[-1][-1]
    # breakpoint()
    for i in range(n):

        completion = llm.complete(prompt, max_new=600, temp=1)[0]
        # ans = 'Here are some facts and rules:'.join(ans.split('Here are some facts and rules:')[:5])
        # if len(ans.split('Facts')) > 7:
        #     ans = 'Facts'.join(ans.split('Facts')[:5])
        # ans_prompt = ans + "Therefore, the final answer (Yes/No) is: "
        # yn = llm.yn([ans_prompt]).values
        # nli = torch.tensor(yn[0], yn[2])
        ans = completion.split('***')[n_fewshot]

        try: lines = ans.split('\n')
        except: breakpoint()
        i = -1
        notherefore=False
        try:
            while 'Therefore' not in lines[i]:
                i -= 1
                if -1*i == len(lines):
                    notherefore=True
                    break
        except:
            breakpoint()

        # if 'Therefore' in lines[i]:
        # notherefore = True
        if not notherefore:
            if 'Yes' in lines[i]:
                nli = [1,0]
            elif 'No' in lines[i]:
                nli = [0,1]
            else:
                yn = llm.yn([ans + '\n So, is the statement true? Answer: '], maybe=True)
                nli = torch.tensor([yn[0] + yn[1]/2, yn[2] + yn[1]/2])
                print('had to yn', yn)
        else:
            yn = llm.yn([ans + '\n So, is the statement true? Answer: '], maybe=True)
            nli = torch.tensor([yn[0] + yn[1]/2, yn[2] + yn[1]/2])
            print('had to yn', yn)


        votes[torch.tensor(nli).argmax()] += 1
    print(completion)
    print('====================================')
    print(ans)
    print(votes)
    
    sc_ans = answers[votes.argmax()]
    label = labels[key].strip(' ')
    if sc_ans == label: 
        acc_70 += 1
    preds_70[key] = sc_ans
    cot_flag=True
    solout = value[1]
    if len(solout['pos'])==0 and len(solout['neg']) > 0:
            if cot_flag == True:
                preds_8[key] = 'true'
            else:
                preds_8[key] = 'false'
        
            # if preds[name] != labels[name]:
            #       
    elif len(solout['pos'])>0 and len(solout['neg']) == 0:
            if cot_flag == True:
                preds_8[key] = 'false'
    if preds_8[key] == label:
        acc_8 += 1
    print(sc_ans, preds_8[key], label, sc_ans==label)
    print('70 acc', acc_70/len(preds_70))
    print('8 acc', acc_8/len(preds_8))
    print(key)
    


In [ ]:
nvotes

In [ ]:
len(preds_70)

In [ ]:
n_votes_t = torch.tensor(n_votes)/20
v_freq = {}
apv = {}
for i in range(10, 21):
    apv[i] = 0
    v_freq[i] = 0
for v in n_votes:
    if v > 10:
        apv[v] += 1
        v_freq[v] += 1
    elif v <= 10:
        v_freq[20-v] += 1

aapv = []
for key, value in apv.items():
    aapv.append(value/v_freq[key])

line = []
for i in np.arange(0, 1.05, 0.05):
    line.append(i)
plt.plot(np.arange(0.55, 1.05, 0.05), aapv[1:], label='70B SC-20 calibration')
plt.plot(np.arange(0, 1.05, 0.05),line, color='black')
plt.ylim(0,1)
plt.xlim(0, 1)
plt.legend()

In [ ]:
predvars = {}
for key, value in temp_outs.items():
    predvars[key] = []
    for i in range(2, int(((len(value[0])-1))/3)+2, 3):
        predvars[key].append(value[0][i])

In [ ]:
temp_outs_str

In [ ]:
varlabels = json.load(open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/core_labels.json', 'r'))
fullrules = json.load(open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/core_fullrules.json', 'r'))

In [ ]:
data[579]

In [ ]:
varlabels.keys()

In [ ]:
print(varlabels['579'])
print(fullrules['579'])
print(predvars['clutrr579.cnf'])
print(temp_outs['clutrr579.cnf'])

In [ ]:
temp_outs['clutrr579.cnf'][6][0].split('Facts:')[5]